# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. This dataset describes protein abundance, modifications, and sequences from human samples, including fields such as accession, description, coverage, molecular weight, and calculated parameters across several experiments.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata via Croissant schema
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")
print(f"Cite as: {getattr(dataset.metadata, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate all record sets, and for each, show their fields and columns by `@id`.

In [ ]:
# Helper: collect all record sets with their @id, title, and fields
record_sets = getattr(dataset.metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the schema. Attempting to resolve via hasPart/parts...")
    # Workaround: sometimes 'hasPart' or 'distribution' contains main tables/files
    record_sets = []
    for dist in getattr(dataset.metadata, 'distribution', []):
        if hasattr(dist, 'recordSet'):
            record_sets.extend(dist.recordSet if isinstance(dist.recordSet, list) else [dist.recordSet])

if not record_sets:
    print("Warning: No explicit record sets declared.\n")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs}")
        if isinstance(rs, dict):
            fields = getattr(rs, 'field', getattr(rs, 'fields', None))
            if fields:
                print("Fields and Columns:")
                for fld in fields:
                    print(f"  Field @id: {fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld}")
                    if isinstance(fld, dict):
                        if hasattr(fld, 'column'):
                            columns = fld.column if isinstance(fld.column, list) else [fld.column]
                            for col in columns:
                                print(f"     Column @id: {col['@id'] if isinstance(col, dict) and '@id' in col else col}")
            else:
                print("  (No fields enumerated in @id path)")
        print()

# Try to enumerate record sets by using metadata API
print("Record sets as found by mlcroissant:")
for rec in dataset.record_sets:
    print(f" - Record set: {rec['@id']}")

### Display first few records of a chosen record set

Let's enumerate the available record set `@id`s, then load the first few records from one using the `mlcroissant.Dataset.records` interface.

**Note:** All references to entities use their `@id`.

In [ ]:
# List all available record set @id's
record_set_ids = [rec['@id'] for rec in dataset.record_sets]
print("Available record set @id's:")
print(record_set_ids)

if record_set_ids:
    # Display first 3 records from the first record set as an overview
    print(f"\nSample records from record set: {record_set_ids[0]}")
    for idx, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(rec)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

We map each record set by its `@id` to a pandas DataFrame for further processing.

To reference specific fields and numeric columns later, we'll inspect the schema first for `@id`s and use them as keys.

In [ ]:
dataframes = {}

# For demonstration, extract ALL record sets by their @id
for recid in record_set_ids:
    records = list(dataset.records(record_set=recid))
    if records:
        df = pd.DataFrame(records)
        dataframes[recid] = df
        print(f"Loaded {len(df)} rows from record set {recid}.")
    else:
        print(f"Warning: Record set {recid} has no data.")

# Display the available columns from the primary record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set is not None:
    print(f"\nColumns in main record set ({main_record_set}):\n{dataframes[main_record_set].columns.tolist()}")
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Let's apply some basic data processing steps using columns by their `@id`s.

Typical tasks include:
- Filtering records on a numeric field
- Normalizing numeric data
- Grouping by a categorical field

**Note:** Update the numeric and grouping field `@id` as appropriate.

In [ ]:
# Choose record set
rs_id = main_record_set
df = dataframes[rs_id]

# Identify a numeric field by its @id (inspect columns if unsure):
print(f"All columns in record set {rs_id}:\n", df.columns.tolist())

# Example: Let's try to find likely numeric columns by name keying on common protein fields
numeric_candidates = [col for col in df.columns if 'intensity' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower() or 'abundance' in col.lower()]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Fallback: just take the first column
    numeric_field = df.columns[0]

print(f"Using {numeric_field} for numeric analysis.")

# Filter out zero/NaN values and only keep numeric values
df_num = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df_num.mean() if df_num.notna().any() else 0
filtered_idx = df_num > threshold
filtered_df = df[filtered_idx].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")

filtered_df[f"{numeric_field}_normalized"] = (df_num[filtered_idx] - df_num[filtered_idx].mean()) / df_num[filtered_idx].std()
print(f"Normalized values for {numeric_field} (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field, e.g. 'description', 'sample', 'condition', or similar. Choose by inspecting column names.
group_candidates = [col for col in df.columns if 'desc' in col.lower() or 'sample' in col.lower() or 'condition' in col.lower()]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name='mean').reset_index()
    print(grouped_df.head())
else:
    group_field = None
    print("No suitable group field found for grouping.")

## 5. Visualization

Let's plot distributions and relationships between numerical and categorical fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping field exists, show average by group
if group_field is not None:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field, y='mean')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded and explored the protein abundance and metadata record sets from the FAIR² mass spectrometry dataset using the `mlcroissant` interface.
- Data was examined by referencing all fields and columns by their unique `@id` values.
- Numeric columns (e.g., abundance, coverage, counts) were filtered, normalized, and optionally grouped by relevant categorical fields for summary statistics.
- Visualizations help illustrate protein distribution and per-group means.

You can adapt this template to analyze specific investigation questions by filtering for fields of interest and creating advanced plots.